# Acceptance: the solve takes your diffrax objects

**This is a sign-off page, not a tutorial.** It exercises roadmap step 29
(composability package `CP1`, findings `CX1`–`CX5`): `run(solver=)` now takes
a *backend* object. `summer4.solvers.Diffrax` holds your own diffrax solver,
stepsize controller, adjoint and event; `Euler()` is summer4's fixed-step
stepper; anything implementing `SolverBackend` works. `SolveSpec` is public,
`run` is a documented composition of public steps, and a `Result` carries
`final_state` / `final_time`. Nothing here asserts; the automated checks live
in `tests/test_solve_composable.py`.

Run every cell top to bottom (under a minute). Sign off only if every
**What to check** box holds.

| | You are checking that… |
|---|---|
| A | the stepsize controller you build is the one diffrax uses, and the `rtol` shorthand is the same solve |
| B | the adjoint you choose is the one used for gradients |
| C | an event you build stops the run, and the result says so |
| D | a run continues exactly from another run's `final_state` |
| E | `run` is exactly the public steps its docstring lists |
| F | a backend you write yourself plugs into `run` |

The model is a three-compartment `S → I → R` with rates as parameters.

In [ ]:
import inspect

import diffrax
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from summer4 import (
    Compartments,
    FlowModel,
    Param,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    TransitionFlow,
)
from summer4.results.plan import EVERYTHING
from summer4.solvers import Diffrax, Euler, SolveSpec, resolve_solver

state = Property("state", ("S", "I", "R"))
model = FlowModel(PropertyMap.from_property(state))
model.add_flow(TransitionFlow("infection", state["S"], state["I"], Param("infection")))
model.add_flow(TransitionFlow("recovery", state["I"], state["R"], Param("recovery")))
cm = model.compile()
y0 = PropertyData.wrap(cm.pmap, np.array([999.0, 1.0, 0.0]))
params = {"infection": 0.35, "recovery": 0.1}
daily = SavePlan(requests={"I": SaveRequest(Compartments(where=state["I"]))})


def infectious(result):
    return np.asarray(result["I"].values.data).reshape(-1)

## A. Your stepsize controller reaches diffrax

Four solves of the same 60 days with Dopri5:

1. the shorthand `solver="dopri5", rtol=1e-6, atol=1e-8`;
2. the same controller spelled out: `Diffrax(diffrax.Dopri5(), stepsize_controller=diffrax.PIDController(rtol=1e-6, atol=1e-8))`;
3. that controller with `dtmax=0.5` added — an option `run` never had a keyword for;
4. no controller at all: `Diffrax(diffrax.Dopri5())`, a constant step of `dt=1`.

The table shows each solve's accepted step count and how far its infectious
curve is from solve 1; the plot overlays the four curves.

In [ ]:
window = dict(t0=0.0, t1=60.0, dt=1.0, save=daily)
pid = diffrax.PIDController(rtol=1e-6, atol=1e-8)
solves = {
    "1. shorthand rtol/atol": cm.run(params, y0, solver="dopri5", rtol=1e-6, atol=1e-8, **window),
    "2. Diffrax + PIDController": cm.run(
        params, y0, solver=Diffrax(diffrax.Dopri5(), stepsize_controller=pid), **window
    ),
    "3. ... + dtmax=0.5": cm.run(
        params,
        y0,
        solver=Diffrax(
            diffrax.Dopri5(),
            stepsize_controller=diffrax.PIDController(rtol=1e-6, atol=1e-8, dtmax=0.5),
        ),
        **window,
    ),
    "4. Diffrax, constant step": cm.run(params, y0, solver=Diffrax(diffrax.Dopri5()), **window),
}
reference = infectious(solves["1. shorthand rtol/atol"])
table_a = pd.DataFrame(
    {
        "solver": [res.solver.solver for res in solves.values()],
        "steps": [int(res.solver.num_steps) for res in solves.values()],
        "max |I - solve 1|": [float(np.max(np.abs(infectious(res) - reference))) for res in solves.values()],
    },
    index=pd.Index(list(solves), name="solve"),
)

curves = pd.DataFrame({name: infectious(res) for name, res in solves.items()}, index=pd.Index(np.arange(61.0), name="day"))
fig = curves.plot(title="A. Infectious, four ways to configure Dopri5")
fig.update_layout(xaxis_title="day", yaxis_title="infectious", legend_title="solve")
fig.show()
table_a

> **What to check**
>
> - Rows 1 and 2 have the **same** step count and `max |I - solve 1|` is exactly `0.0`: the shorthand builds that controller.
> - Row 3 takes **at least 120** steps (60 days, no step above 0.5) and still agrees with row 1 to well under 1 person.
> - Row 4 takes exactly **60** steps (constant `dt=1`).
> - All four curves lie on top of each other in the plot.

## B. The adjoint you choose is used for gradients

The loss is the sum of daily infectious counts over 40 days; the gradient is
with respect to the infection rate. Three adjoints, each passed as
`Diffrax(..., adjoint=...)`, are each asked for the gradient twice — by
reverse mode (`jax.grad`) and by forward mode (`jax.jacfwd`):

- `None` — diffrax's default `RecursiveCheckpointAdjoint`, reverse mode only;
- `DirectAdjoint()` — differentiates straight through the solver loop;
- `ForwardMode()` — forward mode only.

Which cells succeed depends on the adjoint, so the grid shows whether the
adjoint you passed is the one diffrax used. A central finite difference is the
reference. (`BacksolveAdjoint` is not offered: diffrax refuses it with the
`SaveAt(subs=...)` summer4 saves through.)

In [ ]:
def loss_with(adjoint):
    backend = Diffrax(
        diffrax.Tsit5(),
        stepsize_controller=diffrax.PIDController(rtol=1e-8, atol=1e-10),
        adjoint=adjoint,
    )
    plan = SavePlan(requests={"I": SaveRequest(Compartments(where=state["I"]), ts=np.arange(0.0, 41.0))})

    def loss(beta):
        res = cm.run({"infection": beta, "recovery": 0.1}, y0, t0=0.0, t1=40.0, dt=0.5, save=plan, solver=backend)
        return jnp.sum(res["I"].values.data)

    return loss


def attempt(transform, loss):
    try:
        return f"{float(transform(loss)(0.35)):.5f}"
    except Exception as exc:
        return f"refused: {type(exc).__name__}"


adjoints = {
    "None (RecursiveCheckpointAdjoint)": None,
    "DirectAdjoint()": diffrax.DirectAdjoint(),
    "ForwardMode()": diffrax.ForwardMode(),
}
grid = pd.DataFrame(
    {
        "jax.grad (reverse)": {name: attempt(jax.grad, loss_with(adj)) for name, adj in adjoints.items()},
        "jax.jacfwd (forward)": {name: attempt(jax.jacfwd, loss_with(adj)) for name, adj in adjoints.items()},
    }
).rename_axis("adjoint")

f = loss_with(None)
h = 1e-2  # float32 losses: a smaller step is dominated by rounding
finite_difference = float((f(0.35 + h) - f(0.35 - h)) / (2 * h))
values = {
    "default, reverse": float(jax.grad(loss_with(None))(0.35)),
    "DirectAdjoint, reverse": float(jax.grad(loss_with(diffrax.DirectAdjoint()))(0.35)),
    "ForwardMode, forward": float(jax.jacfwd(loss_with(diffrax.ForwardMode()))(0.35)),
    "finite difference": finite_difference,
}
fig = pd.Series(values).plot.bar(title="B. The gradients that succeed agree")
fig.update_layout(xaxis_title="", yaxis_title="d loss / d infection", showlegend=False)
fig.show()
print(f"finite difference: {finite_difference:.5f}")
grid

> **What to check**
>
> - In the grid, the default adjoint **succeeds** under `jax.grad` and is **refused** under `jax.jacfwd`.
> - `ForwardMode()` is the other way round: **refused** under `jax.grad`, **succeeds** under `jax.jacfwd`. The flip is what shows `run` used the adjoint you passed.
> - `DirectAdjoint()` succeeds under both: it differentiates the solver loop itself.
> - Every number that succeeded agrees with the others to about 5 significant figures, and with the finite difference to about 1e-3 relative; the four bars are the same height.

## C. An event you build stops the run

`diffrax.Event` with a condition on the state: stop once the epidemic has
waned — past day 7 and fewer than 100 infectious. The dashed lines mark the
threshold and the `final_time` the result reports.

In [ ]:
def epidemic_over(t, y, args, **kwargs):
    return (t > 7.0) & (y[1] < 100.0)


ts = np.arange(0.0, 81.0)
stopped = cm.run(
    params, y0, t0=0.0, t1=80.0, dt=0.1,
    save=SavePlan(requests={"I": SaveRequest(Compartments(where=state["I"]), ts=ts)}),
    solver=Diffrax(diffrax.Tsit5(), event=diffrax.Event(epidemic_over)),
)
saved = infectious(stopped)
trajectory = pd.Series(np.where(np.isfinite(saved), saved, np.nan), index=pd.Index(ts, name="day"), name="infectious")
fig = trajectory.plot(title="C. The run stops when the epidemic has waned")
fig.add_hline(y=100.0, line_dash="dash", annotation_text="100 infectious")
fig.add_vline(x=float(stopped.final_time), line_dash="dot", annotation_text="final_time")
fig.update_layout(xaxis_title="day", yaxis_title="infectious", showlegend=False)
fig.show()

pd.Series(
    {
        "final_time": float(stopped.final_time),
        "infectious at final_time": float(np.asarray(stopped.final_state.data)[1]),
        "solver.ok": bool(stopped.solver.ok),
        "solver.event": bool(stopped.solver.event),
        "saves after final_time that are inf": int(np.sum(np.isinf(saved[ts > float(stopped.final_time)]))),
        "saves after final_time": int(np.sum(ts > float(stopped.final_time))),
    },
    name="C",
)

> **What to check**
>
> - The curve rises, peaks, falls, and ends at the dotted `final_time` line, just below the dashed 100 line — well before day 80.
> - The table shows `infectious at final_time` just under 100, `solver.ok` **True** (an event stop is a success) and `solver.event` **True**.
> - Every save after `final_time` is `inf` (the last two rows are equal): diffrax's marker for times the run never reached.

## D. A run continues from another run's `final_state`

One Euler run over 80 days, and the same 80 days as two runs: 0–40, then
40–80 starting from the first run's `final_state` at its `final_time`.

In [ ]:
whole = cm.run(params, y0, t0=0.0, t1=80.0, dt=0.5, save=daily, solver="euler")
first = cm.run(params, y0, t0=0.0, t1=40.0, dt=0.5, save=daily, solver="euler")
second = cm.run(
    params, first.final_state, t0=float(first.final_time), t1=80.0, dt=0.5, save=daily, solver="euler"
)

legs = pd.DataFrame(
    {
        "one run": pd.Series(infectious(whole), index=np.asarray(whole["I"].times.values)),
        "leg 1": pd.Series(infectious(first), index=np.asarray(first["I"].times.values)),
        "leg 2 (from leg 1's final_state)": pd.Series(infectious(second), index=np.asarray(second["I"].times.values)),
    }
).rename_axis("day")
fig = legs.plot(title="D. Two legs joined at day 40 against one run")
fig.update_layout(xaxis_title="day", yaxis_title="infectious", legend_title="")
fig.show()

pd.Series(
    {
        "first.final_time": float(first.final_time),
        "first.final_state": np.round(np.asarray(first.final_state.data), 3).tolist(),
        "max |leg 2 - one run|": float(np.max(np.abs(infectious(second) - infectious(whole)[80:]))),
        "max |final states differ|": float(np.max(np.abs(np.asarray(second.final_state.data) - np.asarray(whole.final_state.data)))),
    },
    name="D",
)

> **What to check**
>
> - `first.final_time` is **40.0** and `first.final_state` is a three-number state that sums to 1000.
> - In the plot, leg 1 and leg 2 meet at day 40 and lie exactly on the one-run curve.
> - Both differences in the table are zero or at float32 rounding (below about 1e-3 people).

## E. `run` is exactly its documented expansion

`run`'s docstring lists the public steps it is made of. The cell prints that
list, performs the steps by hand, and compares every saved output and the
final state with a plain `run` call, for both built-in backends.

In [ ]:
doc = inspect.getdoc(cm.run)
print(doc[doc.index("Exactly equivalent to"):])

def raw(values):
    return np.asarray(getattr(values, "data", values))


rows = {}
for solver in ("euler", "dopri5"):
    by_run = cm.run(params, y0, t0=0.0, t1=20.0, dt=0.5, solver=solver)
    prepared = cm.prepare(params)
    plan = cm.expand(EVERYTHING)
    spec = SolveSpec.window(t0=0.0, t1=20.0, dt=0.5, dense=plan.dense)
    backend = resolve_solver(solver)
    out = backend.solve(cm, y0=y0, params=prepared, spec=spec, plan=plan)
    by_hand = cm.assemble_result(plan, out, spec=spec)
    rows[solver] = {
        "backend": type(backend).__name__,
        "outputs compared": len(by_run.outputs),
        "largest difference": max(
            float(np.max(np.abs(raw(by_run[k].values) - raw(by_hand[k].values))))
            for k in by_run.outputs
        ),
        "final_state difference": float(np.max(np.abs(by_run.final_state.data - by_hand.final_state.data))),
    }
pd.DataFrame(rows).T

> **What to check**
>
> - The printed expansion lists: `prepare` → `initial_state` → `expand` → `SolveSpec.window` → `resolve_solver` → `backend.solve` → `assemble_result`.
> - `euler` resolves to the `Euler` backend and `dopri5` to `Diffrax`.
> - Every `largest difference` and `final_state difference` is exactly **0.0**.

## F. A backend you write plugs into `run`

A backend is anything with a `name` and a `solve(model, *, y0, params, spec,
plan)` method returning a `SolveOutput`. This one records the `SolveSpec` it
is handed and then delegates to `Euler()` — the smallest possible custom
backend.

In [ ]:
class Recording:
    name = "recording"

    def __init__(self):
        self.specs = []

    def solve(self, model, *, y0, params, spec, plan):
        self.specs.append(spec)
        return Euler().solve(model, y0=y0, params=params, spec=spec, plan=plan)


recording = Recording()
mine = cm.run(params, y0, t0=0.0, t1=10.0, dt=0.5, save=daily, solver=recording)
euler = cm.run(params, y0, t0=0.0, t1=10.0, dt=0.5, save=daily, solver="euler")
print(recording.specs)
pd.Series(
    {
        "solve calls recorded": len(recording.specs),
        "max |mine - euler|": float(np.max(np.abs(infectious(mine) - infectious(euler)))),
    },
    name="F",
    dtype=object,
)

> **What to check**
>
> - Exactly **one** `SolveSpec` was recorded, reading `t0=0.0, t1=10.0, steps=20, dt=0.5`.
> - `max |mine - euler|` is **0.0**: `run` used your backend and nothing else.